In [1]:
import pandas as pd
import numpy as np
import os
import json
import requests
from pyjstat import pyjstat
from collections import OrderedDict

Eurostat queries based on query builder: https://ec.europa.eu/eurostat/web/json-and-unicode-web-services/getting-started/query-builder

In [2]:
dir_in = "../source_data/Eurostat/"
dir_out = "../parsed_data/"

In [3]:
map_country_ISO = {
    "Belgium" : "BE",
    "Bulgaria" : "BG",
    "Czechia" : "CZ",
    "Denmark" : "DK",
    "Germany (until 1990 former territory of the FRG)" : "DE",
    "Estonia" : "EE",
    "Ireland" : "IE",
    "Greece" : "GR",
    "Spain" : "ES",
    "France" : "FR",
    "Croatia" : "HR",
    "Italy" : "IT",
    "Latvia" : "LV",
    "Lithuania" : "LT",
    "Luxembourg" : "LU",
    "Hungary" : "HU",
    "Netherlands" : "NL",
    "Austria" : "AT",
    "Poland" : "PL",
    "Portugal" : "PT",
    "Romania" : "RO",
    "Slovenia" : "SI",
    "Slovakia" : "SK",
    "Finland" : "FI",
    "Sweden" : "SE",
    "United Kingdom" : "GB",
    "Norway" : "NO"
}

In [4]:
map_tech =	{'Anthracite' : 'HardCoal',
			'Biogases' : 'Biomass',
			'Blended bio jet kerosene' : 'Biomass',
			'Blended biodiesels' : 'Biomass',
			'Blended biogasoline' : 'Biomass',
			'Brown coal briquettes' : 'Other',
			'Charcoal' : 'Biomass',
			'Coal tar' : 'Other',
			'Coke oven coke' : 'Other',
			'Coking coal' : 'HardCoal',
			'Gas coke' : 'Other',
			'Geothermal' : 'Other',
			'Hydro' : 'Hydro',
			'Lignite' : 'Lignite',
			'Manufactured gases' : 'Gas',
			'Natural gas' : 'Gas',
			'Non-renewable waste' : 'Other',
			'Nuclear heat' : 'Nuclear',
			'Oil and petroleum products (excluding biofuel portion)' : 'Oil',
			'Oil shale and oil sands' : 'Oil',
			'Other bituminous coal' : 'HardCoal',
			'Other liquid biofuels' : 'Biomass',
			'Patent fuel' : 'Other',
			'Peat and peat products' : 'Other',
			'Primary solid biofuels' : 'Biomass',
			'Pure bio jet kerosene' : 'Biomass',
			'Pure biodiesels' : 'Biomass',
			'Pure biogasoline' : 'Biomass',
			'Renewable municipal waste' : 'Biomass',
			'Solar photovoltaic' : 'Solar',
			'Solar thermal' : 'Solar',
			'Sub-bituminous coal' : 'Lignite',
			'Tide, wave, ocean' : 'Other',
			'Wind' : 'Wind'}

load nrg_bal_c which has more detailed generation data per technology and country in GWh but is on a yearly basis

In [5]:
indicator = 'nrg_bal_c'
dataformat = 'json'

params = dict(
    time={'2016','2017','2018'},
    precision=1,
    geo = {'AT','BE','BG','HR','CZ','DK','EE','FI','FR','DE','GR','HU','IE','IT','LV','LT','LU','NL','PL','PT','RO','SK','SI','ES','SE','CH','GB','NO'},
    unit = 'GWH',
    siec = {'C0110','C0121','C0129','C0210','C0220','G3000','C0350-0370','N900H','O4000XBIO','P1000','RA100','RA200','RA300','RA410','RA420','S2000','RA500','W6210','W6100_6220','C0320','C0311','C0312','C0340','C0330','R5110-5150_W6000RI','R5160','R5210P','R5210B','R5220P','R5220B','R5230P','R5230B','R5290','R5300'},
    nrg_bal = 'GEP'
)

In [6]:
url = 'http://ec.europa.eu/eurostat/wdds/rest/data/v2.1/'+dataformat+'/en/'+indicator+'?'
r = requests.get(url=url, params=params)

In [7]:
df_nrg_bal_c = pd.DataFrame(pyjstat.from_json_stat(r.json(object_pairs_hook=OrderedDict))[0])
df_nrg_bal_c = df_nrg_bal_c.rename(columns={u"time": "time", u"geo": "country",
                                      u"siec": "tech", u"value": "value","unit":"unit"})
df_nrg_bal_c['country'] = df_nrg_bal_c['country'].map(map_country_ISO)
df_nrg_bal_c['tech'] = df_nrg_bal_c['tech'].map(map_tech)
df_nrg_bal_c['MWh'] = df_nrg_bal_c['value'] * 1000
df_nrg_bal_c = df_nrg_bal_c.drop(columns = ['nrg_bal','unit','value'])
df_nrg_bal_c = df_nrg_bal_c.groupby(['tech','country','time']).sum()
df_nrg_bal_c.head()

INFO:numexpr.utils:NumExpr defaulting to 8 threads.


MWh
tech    country time           
Biomass AT      2016  4780348.0
                2017  4920843.0
                2018  4929796.0
        BE      2016  5450000.0
                2017  5777100.0

In [8]:
#export this to CSV
df_nrg_bal_c.to_csv(dir_out + 'generation_yearly_eurostat_nrg_bal_c.csv',index=True)